# Kaggriculture v20 Adaptive Queue — reproducibility experiment

**Objective:** reproduce the exact submitted single-file artifact, verify its hashes and archive layout, then run one official-environment smoke game.

**Source attribution:** the unchanged agent artifact comes from [Tetsu's public adaptive strategy](https://www.kaggle.com/code/tetsutani/adaptive-farming-strategy-for-kaggriculture). An independently published notebook by Salem Ali produced the same `main.py` SHA-256.

## Plan

1. Decode the frozen archive embedded below.
2. Assert the archive and `main.py` SHA-256 values.
3. Assert that the archive contains only root-level `main.py`.
4. Import that extracted file and run it against `starter`.

In [ ]:
from pathlib import Path
import base64
import hashlib
import importlib.util
import io
import tarfile

EXPECTED_ARCHIVE_SHA256 = '1573dd58af311c6448640fb89380b63f88b0d63a10e788d516fb928f005be968'
EXPECTED_MAIN_SHA256 = '997e6bfc5234534e246e945bc61c87858ebf997ab85b0a5c9427dd4ed710f1b6'
ARCHIVE_B64 = 'H4sIAAAAAAAC/+29aZeiTNMw+HyuX6Hl8oKWfQuK6OW+7/tunxYREVAEZHHX3z4sLmhVdfc9M+/MnDOXH7qUjIiMjIyMjIiMpFc4w/0Q9v/zv/PjVT8Bv1//q35e/gYQCLm3XZ+jMBL4H4v3f/4f+CiSjItql//z/8/P+/t7coYLMrMhLSIpsPjeQvCcLPIsS4qWOS9ayjhFiQyhsLIikj/e3pIqwEpgSZm0SCQu8ZxF5BX1B41zM5aULAQuMDLOflhYfMqLH5Y5Lq4YjvqwqO0WtQOOI2cWCVdBf7y1FE5mVqSF5SmGsKgzsZcsHC6K/PYfC07IvOhheQJnLf1sNqOxhzMqwRm5Ukl58C0ukpZ2tlLxSCwvv4k4t9T74Uhc9BAsz5EWQSRXjLLS/pIrdZAqs1tGpi3kTqVuWSu42r281ynvVyQn60y+yaTGsdoty6wVZoZreD8sfZrkLLhO3WJQZySVF5kkZHL2YZFp0iw5VTQiQZPSm0yrfVtU2XGSBadJfGaZM6Ik6wicKhyWlSxTnFhaZN4ib/kbpCorrYstzbCkwZ86tjedb3avdybRzFzt+TEKdXCMLFl4kaF07meK0e+PN3WS35iVwIuy2pVEBvy3XwQv7G/fF+pU3r6vcJm+fT+wzPTt7Q1LpjvFeq1tieqQP1gen0mA1vhjRmoaIZKSBBjkf0yDiPZwRgL/i/CIy0g9Nq26EOe0B0xR/7mQrYMwXiePQsM3kdBKcNqBMpl8GuBSpQJ7mvuKLSXCXqznPNNou8g+sYHqi1BpfGT9sWE0GXCei6Jnuj7sIZgdHdZIgczM3Jc4MMjgRIkdQqfquXYsedzkfuzP0bbJuNEPkzKTDEd4QDrG6BFaZlx5l4yddp2tdAmfzg0GPcdHk0pzQMVbtPVIBKFpognJydi8QQsxT48MTMIuqxcgfHFQQbr02BHebnpzv0R2sY01OIozLoBG0kjRhcFRKY1xdqXJeybsPk7yBDG1h6IHWWKml0QQEbvbdjBs7cmuhbuVIE41EWuxjhHu8iAxejL1VpJLbo3J/dzO47b32qEEHceR0pA7l4n6cViwthVyipbz0T16oqynTLnhI6dJos2uPZMDKe5s08RqP80cctrKwSJrYjTZcudscuDc07soX8CRDdIga7ldp4P33Xa8nC86JnjABR5EVDiHC/2yu7KJbz0713yeJwFf4bKmsdw+xMAyV6UlpraHBp5I6Njx8kig36mW/W2cijupvacTzRZHlfm4dJwk4vXkUCx4rctMIMtuGA/mF4puPkQ6MnFr30t1R/aitDln4BgYBpnUrBr0BDxIt5NeO52Z1QG0SeNJGqE7k9Ms0kr4uFa70clVavaDp7/Y9bEVhE3d+wUoufuKW7EPCXcwnvdEbePZAmxEHcQx6bW7GzvaWyoUGVtq0V9XoGy9a0c6w1OVOIywvtzjYtUNXZFtCiENtp0OC3mbE8EZzqzRnW1cUcbwtDqu+SUHA8NAcevuHoG4ZyIJoK9wTIUbKAOkMZTJe8srLxFiwxV7arOcUPFm9BQdrmKNEpslk1ElHM1cCk7Cf0zG8/NSZnsMZWjv0AWcCjUPh9qtHBs8RKFSi06ByRDhC2drxymIolTauS6XimlnHrDt0qJdUNZB3BneHcXGdiGFoWm+Wdr6+8E4u9m5yPmU3To6kb3HQzsoERuFx+t6atLeJtALTk58pZkCwv7YqATED4HeBE4Xhzv3ho+Iymazm4OZUF9O5FNuJXvpdhsnruNL5307a7YV28tkLtaqjjF0sJyMsivyRE94LB6pw4mDLDTE4dDuYTJZTLEdx6vKvj9VpHyqHSAQNrmgOKwOUfYd3POidStGx9tOIu0cbRJrOYQcG9GpO+uX3ZI7nufOKwRJVpOMkCM6zeQiNCLTeK5fYyM+YF9jDn6rPDyhXK+ETWSimF1y+drINdlEO81zoOTM2bKHUWAPlZDFDm0CNu9KsnmOgHNPVbF4B5JcB+IixeQDNCrlSty87UTWW5hgQWxKXAYNGzbYrhVVN5mloxRx4gUP3ZxEOCog4cC+uiKU/Hke644HiMMr+jA0JGa9iRV2WGOBBM36oXFj0AvN84NFqhUO9MPHXS0YHOZc49Qqkofl+HDQdC8KYGKNBtyBZrI9yJZtrDXmHnn6m33MHTp6lGFlUQ5g1Ag5rw7JMh4eTlYZcjKhWmAhFt27kt5qemGLhUKr0G7ca/bsNDrLQWmnHR+KzBi6NLzDLpyrusq+muxRDtCxtqYSDJ5xOPZcdCk126Hw6NR0HGI7DxZowMUEPhztCr5y1L7gyvCsMBF5ju2sIpttEfPF0Aq1kru1KBmtdMMADF+yikfxsEO8mAuUL0w0ce4DwspJzI+jSziXja/ZeRca9zoNmDiM97nKqZVOF2p2dMZURYDvl/mDe+Wy0+o0g2hJSEeIbXaOpeRkoXZKbueuNbPn15F60ldv7LAwlzyfhf3iuGUzlVw4uYbkRGjR8jCYDdiM6kxP4JkloC7WcDA+782YBVqQmk5fZ+aUNxfKlRcj7V5p0ZPHcgJJhFzkWo5SSiNxSVH5ZJtU+uN9N7dv9FeVZQ9Dsz7qBEbZCezcBz1KJT12gVymXsytAG4z9ShRpcF0ukOv9YwWB0sYHVX8cKQ6GgPh5KoQwcLgDh04IWgCZ/rULtoETrbmZKecB7vYxLFHpV0lEKjXVkHU4U7Mod45FVZoARm5VugWGIrnZoE7jU4UXm/G5VDDGwXtLt5db2bQw4DaQtYIFygeudRi73V065XUtuQ7JEmkRpWaAjnwWT0dzFeY8FRXyqorzlHsHwlbYH+KU46jL9G178ecC00528V+dQdjR3+Tax2JzVg+zhEuV4mx2wUDkS1M9uwQikxPKo0GvFk0EiNvI+PGAjEIQMhzvofPT0t32eYctk+0nYQr+3zCZRf6Vdrt8EB5lQVig1a5s3sid8qbbgPHt+1KT7I6C9kiflJSO3LksQ6DIIcjqt7Z3AU2hLgC/gFbX4mse91a07CizmnMTdD+aPa4iM8qnXHStcBrqRYTS6+OQcI+Qk8zj4B2x75AoLUPKcChFWiEF7VtCJZd1tSEVAJKm52tVtUMVamHQ0n3ob+ZoQgXwwIBEJeOrnq23M8gl0m6JpHSKlKMjMtnQVj25VJ1xAxD/eF2HCmwS/s81M/M7ZIQOsVTfBmTC6Abqp28lYk/C9vWAlsudwahlC+7KAR2udSoE7YtEVFxFtyAH29vB/g+1kjN20MEPecOoNPr86JxFNlvq9zi0ky1D8dYMFHvc+F6tbFcHQcXJN3h8rFk2xpINT2hXtLuCA0vQwYBqcW41fJR1tGCOhw7Nam5lZXOZrCI2MfBC+2d1YjNvHrYw/t14Kz6PzJUAjNRHzMuNRYFoCtDe2bTTLQFIh1qgZFA3TFCoGKgV5DSUDVc8m77S2gucIOl2xXEul5B4IfK8SJMOEeidRGSoVbUb+0AfjLsHiLhvNhF+CA2Avj2sRxPupAqjRyO9Ummup6sh4zbnVoVhuNq/hBwlRhATir7PdHse7d+AvIdhW1hlIA9tYOrsS4Qvl3H40oRrXpEyNvng5zXGj5MsN6osBVTXTy76xEXn+QUQLreSfZoYBv0eweJiJtoDwa4MF20fM0hAhHN8yoge1z+3J6fj6yDas42iQh8p5I7O7xBEszMJlJ0SSP+9bha9M1KA0fZlci0Jy5fdOV3hvC2o16qOusDoJK4VKbtSX5hDXZQWfBNyPyaygfWQivXKp+dVvjcHqRnI97qkEgWsB2pfDjdx3tAriV4l0DPLdRO1Gi3Gu5Ou5TfsegPrIuBtemm3NVD1JleWVO7w4m95OXslE3R46A/7a+i0VOJsPcqfYfXNnZxPXLesFvT80IxzfTlRQEv1auU5zJhMtQi6o82NpK3s24u+I6drKaAwWpDLLqSPzMAKGu5HJfs0cgGRGLDcBtZSYWswKGlfiBSIeH0hSWEKor41cVBEQ4/tSPHtj0Wqq0qIBxaClYvv5uv8yVbfuXlvakZ65imBS5HiMIUKXSzFdjh2iT9s6hwiRzE7UpsLJ2lSjyyqa72HjTc9RzdTQkezpn9rLsuuU+0gPGTjhXdHGjBAZYScwErrF3+pdWNWG2do5c87jeeqXtXpcNkLWKFo0WlE+hLg1g36bUqUNoBCbnktrBQakjCD7bqjaznmGW6qqPIbac5LkVWzzTZ87u6qXG4mK+jgUGTTvfPNn5bE9B61Sat5E08TESWXnFd7+LlQpQOwAt51W3217D9fGGjNBxLAoNgD2muF6sFekmJF+vcm+wcZjZbJV49ZAJFZi16R5u1H5K2aXYfuOTVUHHaRo4IdyGaZWehusZEh8x74GG/O4/yC8i6605TUPjSr2LOmM+W5x3ncHE4vEy9ZV+sLtcihVOGm5KpLRPu0yF/pD2C+fDhsPLZDn5/utP2dmgMq63tByjWP43jMtzxRVNQWYECnJRdgTwH2/jKuYTv8XbEcbKGCo3BvpPuxJCRf1b3pUbihFL85WCr1UlPk8pqmF0dCd+ikYkDbsbaiEiMPRCqFlmmVZccdWV6bPiyPhmPOe0OO9GgM9OW48Ku142co1COt33+zNmTUvf5WHm9LQymWZAqOjpReFwKjMa1kM2Gn1OLVHg3CcxOhxW4LjoIGtsnYhDi3LQhehKCGWTJelIODCKzbnuk69igUeRCxYRVxtN1bkfxdpA5OOmRgPaIHRpp0NP8akEeN+vZqcDZuFx3CiiCZCMdtpgNFRZg0+XLk/AImF36mUlK8AgZ/7lcHvGDIQISm9wZBEQJ3RdjGypXdbp8EmRNDGd+Uiyu7B4qPXGlXZPWdMpOsUTfn0o1e/W4NVsk2hJIZyVgSWN4MSS45hQ58iXoY7a0g848hC3S6tpNhZNlO2n3rC/h8qh7qAX4qG8prZW45Anwfa6X9YWV4dlVP7voDbXfYaVVuLCMnbe1mtxYLvYH2Y3k8+tTF87wpw2lrsr+qCOcowmmGe/CwMYq9cLc0OUrrTwR6Jg4nvqLwWG8bWbXgYrvJLCdYiObRQ98F4WbExe1S4PSadVpRCFH0pXb8vnBqtn29TdsfRbvVPr2ZSlYmbvZQs/mIBBHOknvUspCzotesZ7muqFlha7WEZyMcKWiD2yio6H9qLR3AQHMFcM7olBgicHpOKx4qMrxEIZGB0d2f5lzkxy6P+3oVmF6nGc8rW5nn7ocMgikVCr1DBZKJqAo1OYjynqOgjGvZ1Gjw93KwJ3dL1igsmyG3SPbMOFwtalJDM9WSe/cxgXt9sr5uOqVYd+yWSN5JpraFxTFGqmlUhdltJy6vdHhadddetl4yt13Ogu2I9xvTTrrfmJYiU9dQrkjO8bdadpt9Z7H6eYqjEYSZ2JYbcAFBBxleGkN+yPVvX9LRhh7r8pduiWMqym2CxVNIt40o/o5As6QNvco5iz3/MdmubuMw17EhympGTqiEZJgw/Pp2dcYnMkZUkTzkQPKJ5vHyMVbR+2xOZGqwWVHRVhGqVTG66vR5e4KsSvuva9axZPBtd0fqQ2KKYcruAX6W6kHl0ICADUVq3159i7zXLXl9x1GzX4S9TNpIFumrVbejfcWW3YoY/Vl63ystEW7TEn0esE7hrnQimyh5CCfLiKrXkG4LDchIad6rKE0jGWGy4295eI3AOTP+NhwEj6tCzIX7ndRStlgjkA3jwA+axqanPLN+aB/WBNyaajGdcHGkPalF2cbGZk7W0NvdJ+QJ/SsFB71T+Hp8XIQywUmDA36dVVxAMIRd4S7KQ9ORfHmbDTswV7/yLUrMQdHVFrPm8tRrH9uJcK1YTWeSLiH+aTXv7P5M53y+ajuW9IhEBtHHcBkMpTzUvqYh4o2DMObzJLgGjH2MnblYrmOd56KLtr2Uy294lu9tMdBuQfO3OYY62E0kUXSipdOAQk3vGrhYySdJRuHptfddwjpUiF1mMVyFxlZ2S+BRIHek81wueVfzMm11wZ0RW8imVu3goo73k+6V+54reWsSY1UF4jNkyPPZgGMJuhmly55yiBta41ZryOdDp8XzvlIXZwwBKsbYGx26EUU2Opw1WAfC7nHOSh2ikgRe77RLwxGLAVlU+lhzFMokfE8H6kfMLI8E/xCbTXYX7B8NQUm9h4R4etLvrCJlpaibRyX2EvVDUXL480KLQfxboixbeqzLnEehKHkttqh6/wWUIP2boT2lK0o7OgUNo1Y3p6QV30cr8+9vgXTh+3rSLYwQLpoaChl8lxmXZ10yGIs1rTLrZxTrISRxswJTt2QB3ZXgnIGTOzyfKJ5FCAgnYvQw962H5r2U10k7nelsYjkhQ6BCXshopOZrRzZijDHrdIl70msRLsVJ4mjUThrw+jLsJYqxvJ+PI0xs94SAC+JAZxBYvZ9tUJNzrE0kt1Wdlmi31+eO+w0Y9vHnWGeRqQs1a12+2jCV6BS0bLq563tHS+28RTZlmvRTKTZZPGCT0NEEghsVkimmRh2Qn2FXM0raixNtPMLxX5GrMnJqGctlYq9dHpbreMhnBWESDiZ8FUvrtbUz1SwpgOZ5xyXCkbnD3LDatuOmmdr1zmxD+FsxLGkUniom7H6oyy4F+u4bM2KPXVbmaZGxQjS7+FQ0TdEis2CdY1MZ3kPWG2OmhTc7geWCE26x6GEvPTDSTGQGax3W+WMDgv8cQg0V9kUlaqIShTJuQtM9cSVUzYp3qMjtiXTP7fd+2yK98xZOezxrfMtdBGnyO6BcvmsmXFJ5UwQw/tcGM0llPXW2QxsIWrGs/MVnh9UUdem2gwoxXRIScUu1K6dPOVTuWVhrPjb1uMpwAcrnBSs7Zy2ogy69wAj8XXOvwz5ax0KbdTyLukw4xyNvMozNcra3auCA2V9fQ5Gp7Xq6JDs74R9xslY64gjUIYrcLo6PB0aduE86RFFPxYiJ/5qbxZqHpmuIhZ8lCj7lwnGnysWt7U1WshWhVKcLp3ayqpVWRSOTa/ngA/5oK00DgRrdrns9EUhv3/SqHXHfdRDeJBUgxbmaMcdnBV2rVXC3ZY9+Kbm7u2KvWDGetxl0kzIF4sizhSqhjZd1avbAKWkVJhG0woxIzvpaC6a8h6JiN0+6ExS8RqADVbkqkasqjw0h8ZnKROY2iTM3hLLzXpuRsnisOCAulRu3iqqK6O+EMKIQ/B16XLsAKUJJJXKTrdYIkCSLnsAzu2claHXFQmlsbHkj2c3/CI9a0vx/SAIjKyr6bLVr4Q9wS0esbZ7oVxRZIRMxguOTkz8iMPWWbGSyW/cvQMrZ4JsduXaRWel3ma+2OdOlz4UDu583mxGHNeaFZfvjAfybCEZvfROVIIdd1BWxmqeJRxfh6jtwSHE+Vz3sCPlxmAiKMkacLR5FkorXoa6gIRicjnbThOJDBUGI1sYZFLeOoek86xtK1R84nq0lHMZpeyMVKNnV5S8TE6SwEsJLFhMn/JU3UMtq0RJNTLZITTak0ipLhxVn4HcRLBj89SvjTqb6ikTaB7a5cuODRLqqrbb652A7E7up03fNLB1deujwqziQnGh5fVK7bzD3XLOm8fM3jkYbnnP6MLZGpOBVOBtjcZSGpLuE5s9OKY7Iu1Qpki/f2nbZ7lIeXP0uuY7eMeUrXn0iLtEG1DjmpHAIiKsbc5ceRH0OmohJphrgegcJIoOX720IlrBvXdWQca16WEYzRR2dtGu9JVqelE8ba1rYXyIC/34dFoS7LvcnraF061W1BkMA0E0m27VarYAmYKUPXDK1ma+5Wa0WiquIYusTlJj4nOtJ/30FuCqK0ev3l4WCDe3oYRVfnRpVGhgGS2UJ+iyXz6TTeYwF5sihTe7eYdso7LuPAuh01N+QtAbYs0rK26aKJ/LfhlcctN6kAJD87jSztfH3rJQdofmiR0k2GOuvC8FolQairZqizSI4sOVHLVG4QRHNmNQuUHAvVkR5aHYptB0OwUCCW9tLaZir4xq0+NULrSWZF+INfvB3LnNi/5kZh3PZvOlYQ4Xm1ggQO+JSr2uDMvdfTjTilXD8zO0hCOpbXzKxqbMsrOOTbf1SjAE5cvpmbMWDgvBUmtszZcxetwMBUPeVunkKOXIVeTkHgfHleV2VwU7SHushnF5b4aJjHvtTLk55OclAIkEenvBXZlJRC13ETIrwSaXlLDQaSS8yqjc6E6kileejNrR1ijbJFVLM4M73YrqO1Zn8MIKMPlmEEV4eM/vBWleGzh3DHpgYJq0JTPL+HhWciNZerdsCkxmycGbDeUDq/YiC4XP4P4Ud9cdseU60xgnh3Frd5jPxsRwCJ3BycE82QmemmrUX3ModaRonxU9mNMaPaCDo60v0VImJGVkwtuezNJTF5G9AOE4SJedcixO4I4d1dzCh/LAPdaWMRTwN/OxQL4SKboQZ/sAYnsHcRzFNqG2AzlP6qP6jIfhfmkbRLJTV71bc6Xdmb69GWqxzkb0wMwq2YMdXMWrw7l7lm8pzvTu4BxCnUXA2gyKZ3nvr8wTR0evverTMYZKMVvMjiwymxhLMy4KuWxjYBf11YKj/TLTT3kSCO7Kjuh5Ld+C/dnGMOWSAu50AYtcDhCw5Sa1ZtPfJq2FBisWivyexBVpRs2gghq/JNrFkscjhBn/2K86Wsf6RSjk+7l6PYr1CvtKFu2nKnTiWK1SrW2xJ5CL9npUEmrVZGWbg6VLqiJXjpKnWVlvK1y3wHCxZHjGnSYNEW/YuYo0WEEcGvA4O84wUCVtoRzh3ADrHJ+qEnVwzreCSAMQ27vz2BX0wTkbO12uoyVnuxRqTPPcyjZdoyW80A4XS9PQqNmslGR/nd6z1TR8Ro75cHfmGPnKgm8xt0K1Q4VWok7//rxZVdLN5kbdzeuVLXk4R8Nblk/jw5NNWpzJTTfQCrGI2Kw3zwSaFQe7smcH4DUHuSxdkjV/y+5eLkKQp40Snf0WXDu3wMlGyUcnOJx3p1L+LGzGYqM0aU8qqxZd2suVUt0ZaPKn1lYUTlP7uWV3oJcWsTwefDuHNblPSRs+Gyj4Z1jGv4AquXYb7IerbgHIocq+VvWHEHCukOd+tDToNl0SWmb8vIJnoP2gsOR6/k3JWmrnRyUH6ClGKLFerR7j8byLWXb78TqL9ke1dWs1GBCHGn7q9ATEPwt7U21Xp+HZnJ12aFtEbChK5lG2kWtu69G+f5tXh1hrWj3SnAeLroVnnTgzC3DRyc4dzZMn7wpCbMy6CjoH2YQspodbGRaWTK0f6pSJcNUZWaLyBp73FzW8r4Qu3trUSlVt+z0tpHtF13nhcni9ttSJ9cyZo8C7HIB9V0nzvWMh7KzI9fZp4jmB6QsXoCqu/lYouraBigSv2YUknYaZpYA69q1mBPZt0hE3eSiuIm372RkdNKUhjLQLwYxvjOa34UAsOeFVGbDWbn7rA1L2anKrrlbncipuPLt9dEBMzy40dODtkzpNM55trHyZZqMlOt4Pb1NQOGlHpmGmkQpW5xzYPtK2fpZTkJaV7JyssRC0H9tqqYg0TseGTcYT6U6Yg7vg8IkhfzM8swXw+ASLMLBXsRXFWB+p7ur+7niakzaXBd6MHUIrFx+WB+PcoBDMZ+SsIy+5ixFU4rhhbVthDktpsbnIaJLMs+U1tK36L3GZ66TzfXeja2010BLRiJ6LVWvH2xJoNKROq9+b6taScLfdtias52TXvTiOYYgNZPz7FuWv5hup1G47mDjXR1nJY64aZ+WEVPhYKzTDDc+CXBSEemMmJSNO1jMt9YLzbSQ8bXRhrMkJ9CpeUvKwc3xS9x4itU+mfapjNS1kxvtUPT0eFdupncPNJeOx6C6OpqSIRGag2KzMjItee74N1lKNSbVWmi2LNJuvXJgFV0/5mo5waa7Ih/CuX1nya2FpRXpQtEu06w1mAtoLPVcjsCwnG+ARJBY7Qei7Sj7Eui701mucDjjgVrfsr+0dDFDAN2wWEH15LMoQOXYrxQfDWIdfYuSGFQhuRyhSyomPpyAjYqeZt79PBiNuwOsO7FayFSjQZTZ/PPvhrj9YOB67Rx/Q2/pOhIefUMVksBVHq/7JJmKby3bfNhM4DXMBVBHbYDVY2Ud7u43iKxG5RNRlF2epGDeJTiqJfDQntgZVd8wxbnHsrBCadurLcCZVBH2s6O8CzZq3noiNUSK/kdupciDvkwJTtiHCm5ySrITEkS0JFv1VStynkwvZF7Emxyc6uBDaQGNGB7F+ZVevz6fzSbuTww/HeCuwkija2QEm0YaCBusM6KNIqbNsBfjqNtEiofXQfqk4VnF2UsSoM8oo9CCmOoDQOV1IistRa+EfDTy+Za58dM4Qf708pi7ugCIexUIlUkinbI4zNcV2M59vVLXiQNZBWY8wHDpgoaDfuQ4HPJz90mrDmeHGS806s6VSHTtd1XAr1TwxzR3Hpqv5QU22Dxtg1pe2yptRyN8jLoJ15A+6+jO7v71hDucjuvBhS25WpZyzNmkbhfkikFViUZRvMHmA8NcT7VPcOii3Qg6ePTUBPxeOirGWf11xezcXT2EXpMWB7DqGpBKxhlJxpFaszfZSqj/CqMm44kszY35dbHoiWx4B0FI8l1gFZSnl69HrWUukGZS6TGJ83+2pJevbwEakOsei/xJEOwm5VEig1nny3N03ptYsAGx5ebZPxUIFacR10AsBb9deohnqJXNEMu0PQYhqwt1QqOob0gzXzC3bOcQn5Z3bqCzYxp26teYI9LI7cDqh08gMyBW2nO9CDSk3WI3ENkDx0A1u/GG3Yx2zdm15NkwBHA9mnE7B2WKO+G5XRQ9VaDmiJhMSAeBSbOL2xneF7gEJ7vPF5WpJ4ai1X51MMw2Xr5rIZZn6gM4mMkowCtsahVOP2YCdOeTdCTM6NxbBfm0MJlUXue8IuTJCVkgtI8dSdZp0eNlQuoI3CNFfyR+TVgoO2KTxqO3k9lJuEy2Hu9MaC+FUujroDNOnRFx1/vOUvddBCmQlF8ZFr4feHtecD4bpJTBimim7LbMf1BLVw2A1mCWaitvarHZmVTjGdS/Hba966oJhIJatockYvIMDUcpuoyJy5IDvsYzXHhnzfCWRXIkRNG6rHeCaFz/WJwHKmYI9jUT7eKRc7kRfdXq7nIe+OOuxYSUxJuPdXtoX/l8g+ONaXPSuyHNP8B0E3zBsQ4oSw3MYZola3vFrSZvnWvjl8amxBvT+9oY1WsV0FsupvnFLBYTesEy2mqxlsGSlUUiqT7w/YOQNqyZb5WwHayRbyapW8XR8s6if934hm+y8/2MBYOTDAmm1ih8Wv/bPu7QW5fcPFTuo/mB5Sv8Ogx8GXjrZatV1RJ8JEfGaYc1E0Btip15NduoaYsB7R4T1HllGq0TTwf1m3MANt91pJfupbKs11PAh+EEAemYZNVODHgSq2Uq9Zoz2gevzfuZa/ep7oGXzeQ3JjOODPzP8SUjVYqWsc2oaKgQ/ySXwDaf9er2iM+o1DxL5mtF7h7lsq1OsFEfZlt6t9w8SNv9USZzfsHah3lAVqp7ppjsmLUkly1lD6LosPm56c+u3URyNkjqyDqOP++M+15/AU61uTfXs2w1Dg55pfjzN8w1lmGzVsHan3srqGLp4Pm6Nmv6nW9lk9cGCicSH5c7QC9fqekgncwbFq0Lfabar9XqnUMz+luRd7slWNdtqXxeZweFtMDfCZnE8D1CTe7ZSSaYqWVXksiKwJPC8XlVrUCk2u8VMUitYxOqtTFZb68DTWvyw3ARpUoOPm9LfpfBpKX0xUR9XBfx4U3vWKlVV0Sc7GndH7z+W41lVK+3P+drYyjYqyaEKk21oShPU9KiY6zyQ9D41zHcWl2RMkklBFZIHUvuZKeTr73edtsEo9N8inTWDmM1WGx0sW9NEmlEZ6IgK+XieaxmFn2oD/MP7eF5NDrBUspMuqA2+l+dpVYZZLFNUh1RLa2MKmNqLtasRbmnzo5tc73NzrtvptrJYs5usdYqdoQrifwCoNFsdzXTDXvPDekPrJmg8qxa7VW3GX7XweW6v0wa+vb3NyLkFo0gZ2OCsQn5YluReqzOe4worR2s8R4L/6AJm5hZGYjhJxjmCvAHPGEK+tmsfkdSKbi164w+NppkYqIOpT2VSVDlUv+CyLN4ovau/Vb70Dm/9ETjL4lNVyQ2kzx0Zz7/o5dFu7uMZ7jp0rQ4Ywwmt0Bkw/lz7MX6onGoQ6rZLCtqXK4yFF1U1eurseOfuXav7JkVV01hGkq8YujxuLaCG//O9kWy3339djYOOqVWSSyriTx2TF2eqrMygelm68ZjhLE+kDVSD8i/wl4noCheXqnSfqP6O0hX+hdT5JjGJxFUiU+kqp+vwIV1DOBnQlUltVudUq6kn9X1Dp6X+G1W110KyEmnx3shpIjHgNcJXotpDSZW9zrCJov5cJaiydeXOzIPe+lMj80vjRvtiiVhYkgP0FtDo+XgfCc4yFIfpgrsK4MPyGNhdAb7SER2C3Al6CbzGqNqJzqdpPA9JgR+3qf3Euv74NtLv5/O2JrR+9BZQHdmt/8fC0Jt+kDuZ5GbAz5va6HONafMs4hxFAne+PSZ64LUTg4ef1/5/qaz9nTLq8D//udH+9cs8MwbRuwbR5EyVJ6GVzUvM4WZiaJydq91pTyz/+Y8F/np1ATqYx6La89s3Vbz6U/OT7+CfYcEX3RZEfqGzr/NoTOOTVdBmV1OJL2dZhxBEZoPLpAZkWgnGQ3X6VathGI8r8LW/++6nfVRD9Y9lhe8A1RfTlpRuv4wVBN6BNOHrFs1oVWdAs8aGEl670zw/dRjmXsEfjEyuJAC8jltXK25DcjIvMuTzkntQMUF81mCBlxhNQhryTx1RE87H3QyqCOo4vKqds7gexK8wn1eFoTcKx8jXJafT/cKIfpgs6JXyb+yhTlVm2JcxXtnQGz4PzdBRbSrNKqutGR1Bh4W8Bqw2Hww3I3cfZua1eSE5ReVXlSRgHpVpO9Mtp4ppiRmG5C5RvYOnJtNMmAhoH+0GDsOp/svtwY2Kyv6d4E+d2C9zzxwvm3f2G6i6TDQpfRh+JqhzYuZNsz/wHzhQRbFXe9dU+Ib2U9eDpyfQL9DMD6BhgQZb3G0G1M61B4DXEomqNA2jfp0D7cKQ/nxnfv5z/wv8k4RuwtSYPP5u0X232EyzcZXsyyo7m0dmVguNadNvVSza3vieaWlBxBPXM1KXlKQL8srv0yLWdyL27+ir/nc6+663apIyQYCahr1MqNaL2q2ZDvTrCUITtRaLXCX+c/fcLMmiQmiX5zQBv6frfdUJ0das5uTq8U0hm228PMvX620t1FLBVVmc9cWsMQI+szY3EdeGY1JhjZurb6o3ab8Nm7BUJ+ld90KekTXdekDhHLPC2fcX7flSg3QZiPvPkCK5VkjJMOyaNpllCKtG5rqVv06Az3BRoCd65I4gBVX5O3uBzIqidrOwp6mh/v1vuTSpEaDL88OyYjjgqvB3dvXFedOym+g1B071YczqJpF/WFu6OdRx77f0tFVz5eIZWeR5Tc+uvEBer7pdS8oKuO+OP/Rlpyo8+KwF+IpXOFlDfQxFk/bjYqC+fD/0Hj4pkIH9WX73bn9qA9A8oAcjZpFY3FcSZkflDnp3KZQpyxCYpLqauKZw+pZznTXVpEj3+NyUZ/tzOuAR0z1wtfV1X1T3lfTxtLr0RaXFf2o8fsU15EJoA5HuhtB7M3narGlsnu+bnMhv9ZDh9/vnP0+aoNsJzf/UcJ8iSfXJh74fX73zn+qDX/+8TtXLJmVa4X+p/hoTc4ZkZzrr74TIC5oYrktd/WaYhs/UDIOvGwzgbiJ0SirSOwj+UARBjUHBT4gq1/fNwhDuZ+IPwf/UYX9Z3NGXxX/7TEUSX5oV7aE099jjW5/q41tYhWN5Yqn6NuqSmanRgfwdopFuujKrKsQvs4JIvKhq/LUVBG96dQuxtSvD2Iy5Tt4jwvovo7yrzbzGck/uxy0I9bpcoTdjnHPVdxEZitbMw9eLUDKcke8aoefg8iFvfKr5gHNZ21Q9Rh8apXu7WzXjrgcU9ICCnqA0E6dB4Wr71PAx8A/LVBPogREMZPjXdRTargE+y1W/Bo2pUpXJawAtk8JVtHrsGzUFJ8ZT+RqZmPJuRrz8dt9USUHbH72a4PUfEd2i6pjG9vjIsn1YPJDZybqRf87EaX9+l4u7o3/mSl93Ks0H8z9NpI1mUjDPkgF+FZB+edwQkzmUexLU1bi8JgM/qdY1gn2S4tcTcE+XPYvtPn5DaharwfxLFPBblMgLhkkqd9BfH6ZHuog0aR/PvxnPTDdwR21be/KAb7vo1f/9YjfXfOBnft/BLz1gI6WkBXK/HvsIvjU8/E+ZJ9PKfwzWSDRcjYWK+xQ0aHbhmtfSvSj9JQLa75vjqyXODb93ds1NGs3QL30nj1m8X7q+N6C373w7k7gM2PsyfUDPlFsmSXNT7tgfGiuGiwF+Q98TfaA/e3M3zO9BVKk8CKnRkfcvd8rbMCzRB/4dwJigH7i653Gzq8SfskbXGfyli0b7+vZfK+mXKaO5ollm1ZyxrIThmtZpRoVWQ6/DPTdzM1/u2/Nb0Hx7gcPnDPK9S0lhNfU8nl/V847885n2ry9zpf+8KqWmqA+VVH99VkjtobpHqH3dEvgv3rHO3E8DzJgW7YHev/Hw6oyadFF7ftfE61gNtEemSz8e/j9nHe/x+MvZRCR62zOezifA/xus6Vdm5ivPQl3KvzmQ+R0jdxv1KZX0MslmZ8Ro0qcY8v6OukiucIZjOEp3R75PM77qn9HBf6lXX9oyQ1deVOvK1D3SMevQrfE56DES+Z6vtO2eACX0LJuRD3+4dXfrfs9GGplRQg8dHolRnYrNUnu8zEZ/zY5k4ecyyWlZb14NFGSaZFQtJHfy/V032qt11OcqNEVbDK/uSsvQHktDJOfqJoJb9FfTePQ5utkKnMK1+OKK5zHe7KO93YWU9TxVWH9jze2FNVe62mtrbi+s0alttVfbaO+oUfmmtJfhaEqtvS5HWy4SPid/3Cf41rFmZHwfFliNfM1Rk3LNm/zB9L2m8gzoPwTntxfoRM2OgSwyOIs9r4Krer/AmHVZ9wHuD8BP8f/3du06rrtD8aR8RuMXmvcpin8lY7I9X5yo/uUmqL3MB9NVU2VrzvKq2J9P2o0F8+T268QUUSQ5+RX1mk1XFf3D8jycJ2ybJSnrymOHNEwt0aMtZ8uMJw0Fwmezm5m6Z2lUO8lbVvxGe0nSMzGZVmMALcNkUVcSy6jaSOCcRoZQY0lVudXNXdFfFCVJmqrxhuLqK+7Hq5SfB6baeXM50V8K9TMZs5xdlu8Oyv82D4eLFHlLBn1CMenWixJ/pWQfn/Bf1OwzwOfygI8vmYCuNlONlWfAq/KahHCrPgBfufm0Aq4Dj1yjNj0FblrLnzanP2QzH5g3P++nsavcdNfo70XzX4T61YbyZ7mr24pB/Jn21VzdaRowZst3hXge4xc+qXl0P/+BvL9+H0tdA8yH1/cNuAFp8PCywX72A568W90X0/JKmmU3cjL3ExZjNE/p5f/+ROWWvfh9ts6Uu79lm4vaecY13fynzPO1k/dKPV3OZt5vPqYs4gSJ6W+zux2fG5uX/ug+Plw3lXc/+5bLNYJh1QXU9OPj2ZHXz3h/mQ5StXSuRlT3gW71F6/86Vua3t9vqjGeTuVNwJ+yam+vlI0TcJ2NXw+GIuYzeyPFee/qKqctqXqCxjv+bnL6ziF/FCV8V7zwm/QPha90ST8qtf6c/NFwvsz7PKR7Jftl1kfjbPMp0fPKgFYPpBK5M/ltlufz8bteNPJX59D/Hzh/NqRxHe3Pm3B+vd19Ql1lPjS146THAfKd9oa8pVXMkaZ+PqzN2leLwFA5bS0Zi061ZdC3B8/fnE8/WP8h8AJwZfJRq/XthoJT5M2EesyD+vmuvXpUNclmTnRgdad67tfM0u2MNXpfyQ+C6gjVrYqcvf96ORWFtY1Rox3Rao/cls+liH/T4Xe2zCgsMUT7m+Ox76T3WjlgOn6/DQh8LiHQMsN3TX8qNZDMSnHVg4ca3KdaX99XpdB+e571wUC85uU25C3SNx2+3Bi7HdvcIK6P/3TmPr9DaiHr9ZgfeE91i5UMZjqpUuen1nn/0xH+9QD6uy307a/OkCzXE6qnQ2Jr9HpK9gcODDndzL5mBg3dvpvAu2Ze6wHvE/swiF/r3M/3TDF/sw43Pb/O56/nc3ntKOOlxEB63my+qe16ogH987Spfb/F3A8gcIEEONWWXQsjrrN1OzPTfb8fqvdnhEEGyD19wunbRvReXv51HeknaGn9DaTqPH+HoU3HK472otcfWtOVr89s8dQ3WGoLJPwGDfL+BtELQD+8qh0yoYs4o87Uw7PSRXqXsuGtGlHS1V++B39XgWth1IeFXCsMy0xFRlFBJALX9HtKsvwWmysccftuOM+qzZryG/LaYny/B1BfhbqPU41bzUzE3KHJ8qwElpGVmaYA5h7V6dHYtPznpjdm1nRuH6v1Fj7rCG4TSddXyCY2VGv2kM3bZ1tsZu5p0K/MmaXzO+Y8XzFnRn7Iy2Nm9Ckzq60TczxtjhD13rTU2lUbGEnPBF1z7/98fUJpMnM63Gsx10Me10Ig09nJU9vnM5Qvmq/J6yeleT6pZFaCaj8wieDFa27XzP4tyfzV0MzVvboRefcw3Pz9uuyMxKZ2NH/jBPwcM32dYHo5rfnbGpsrL1rh/lPO+LdpzrfXYjMd2oAzVVfuvygPNadS7whf5EzNySdzT/eKbFO+yLAh+oUfcxbqhr5WePm3uauvLNKnvkEzaVb1XMRXwn9DRV38j1PIt0/KcG9T1959s3keh8fc+V0hjRepY4LaMMP3xsSpO/ucoRQRv/pfWs3ZNUTlt9zzHGtPPslfonnhuZxBAzMXWegQnwsb9Bef35gxT9oLS+86YIMUM7imK7Bfp6L+uXePac6FqG4tV2W/Jpu+JqYy11Zx2uqaK17RVKoGUf9V3IagrldVbs6q1pNe9qEN5x+TYeS1g0jJKDAwXRDT/SoNWLVD4JNPdssQ31BfSx/1zt1RC/Asov88D1abfgC+HY3caJlvG0CPU3mtR823M9+Bu/dKkBrJ/1aKaR3rRY6P2XmM+DGg1/G8dGxWdQPpprm63TJb0heGzMZTh9Jm4zv7ew/8dcBbAvFvjLGO8Xsj/Fd293+zAf2/aBLv2m8YF+3u61+bjse+oleQfzJPX9k4LRjTeTByEKpnwhH7a0Yb0pCBK73/XFkD1W+QSvbJNhqzqS4Jw9F8vufrupG920Ltf4/Q5xrT/jMJ6Tn19DSyv74Y898dpYr8Vs+xPK5s/KWWf1g817jZdLHp5QjKaL/fUnkE0q9HW8wn50pv+mU+7dU4/bryzJzhVYF+aBVxgEhqV7PJqHat8Obpc0ujJli7uKZCaoUWpqJKvYPvCip+aoedgEEC/IJfw9QZQ326mmPQ+M2FnNt/+4GZ/tuP39UEXHODKBT4nSD+rCfa6fcXB8Vf3Jb54iqL4SFd/1eVe/2S9/nk8XZh9lHXcZfKd4VH//xGJZ73p2vnP59sn17D+b3Be+Lu02VdU6SywRn99uOz+cSMrVTLvnx3LqoqiYhr8c2dwm3GVPcehYKGmlxpPoA89+G8lCSp2AbJW7Dwqpraing9X3qF+eYoSSf8XGf5pJm4aqrMlw2fHHsjXf1a+23So2uy+nGq98Uhwtszx5o2fpeOf1Ljp7IgLQP8qFZ5pvb7IsSvEH5nj5+Trg+kP5XzfIHyl4v+dYH//uDhGkdl9T9qk6mg4Y8357687vd8ofaWzXo+EX1cm/184/FP2f4XSo+7sqaW+/XAJU5RLIlJynTFSPo7N1T1FPcCz3BfXIl9KO/b//z7+ffz7+ffz7+ffz//fv79/Pv59/Pv5//Vz/8BUEZ1AgB4AAA='
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'v20_notebook_output'
WORK.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = WORK / 'submission.tar.gz'
MAIN_PATH = WORK / 'main.py'


In [ ]:
archive_bytes = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_ARCHIVE_SHA256
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode='r:gz') as bundle:
    assert bundle.getnames() == ['main.py']
    member = bundle.extractfile('main.py')
    assert member is not None
    main_bytes = member.read()
assert hashlib.sha256(main_bytes).hexdigest() == EXPECTED_MAIN_SHA256
ARCHIVE_PATH.write_bytes(archive_bytes)
MAIN_PATH.write_bytes(main_bytes)
artifact_check = {
    'archive_bytes': len(archive_bytes),
    'main_bytes': len(main_bytes),
    'archive_members': ['main.py'],
    'archive_sha256': EXPECTED_ARCHIVE_SHA256,
    'main_sha256': EXPECTED_MAIN_SHA256,
}
artifact_check


In [ ]:
from kaggle_environments import make

spec = importlib.util.spec_from_file_location('v20_exact_agent', MAIN_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)
env = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': 20260809}, debug=True)
env.run([module.agent, 'starter'])
smoke = {
    'statuses': [str(state.status) for state in env.steps[-1]],
    'rewards': [float(state.reward or 0) for state in env.steps[-1]],
    'steps': len(env.steps),
}
assert smoke['statuses'] == ['DONE', 'DONE']
smoke


## Frozen local promotion evidence

- 50 fixed seeds × both seats against `starter`: **100/100 DONE**.
- Historical-opponent league: **2,010 wins, 0 draws, 0 losses, 0 errors**.
- Project tests: **27/27 passed**.
- Kaggle submission receipt: **55373947**.

These checks establish reproducibility and runtime safety. Dynamic leaderboard Elo remains the authoritative promotion signal.

## Result and next step

The notebook reconstructs the exact submitted bytes without hidden files or private data. The next experiment keeps this route fixed and changes only one observation-driven market timing rule, evaluated from both seats before another submission.